# Notebook 21c: Raw Coordinate Multi-Seed Validation

**Date**: 2026-01-12  
**Goal**: Test learned activations WITHOUT SH encoding using same rigorous multi-seed approach

**Critical Question**: Does SH(L=10) pre-encoding mask learned activation benefits?

---

## Motivation

### Expert Consensus (4/4 experts)
All experts identified this as the **critical missing control**:
- NB19 showed minimal spline gains with SH(L=10)
- Simplicity paper (Teney et al.) used **raw coordinates**, not SH
- Hypothesis: SH pre-smooths signals, removing content learned activations could capture

### Current Evidence
**With SH(L=10)**:
- NB19: Spline +0.36% on elevation (barely significant)
- NB19b: Spline -0.64% on population (ReLU wins)
- NB21/21b: Multi-seed validation in progress

**With Raw Coords** (NB14-16, single seed):
- Raw+ReLU: R² ~0.72-0.73
- Raw+Spline: R² ~0.73-0.74
- Raw+SIREN: R² ~0.74
- **BUT**: No multi-seed validation, can't trust small differences

---

## Key Comparisons

This notebook enables 3 critical analyses:

### 1. Does SH add value?
Compare: Raw+ReLU vs SH+ReLU (from NB21/21b)
- If SH >> Raw: SH encoding is essential
- If SH ≈ Raw: SH is redundant, just adds parameters

### 2. Does SH mask spline gains?
Compare: (Raw+Spline - Raw+ReLU) vs (SH+Spline - SH+ReLU)
- If Raw shows larger spline advantage: **SH masks learned activation benefits**
- If SH shows larger advantage: SH and splines are complementary

### 3. Can learned activations win without SH?
Compare: Raw+Spline vs Raw+ReLU (both with 10 seeds)
- If Spline wins: Confirms simplicity paper predictions
- If ReLU wins: Suggests these tasks genuinely don't benefit from learned acts

---

## Experiments

### Experiment 1: Global Elevation (Raw Coords, Multi-Seed)
- **Config**: 15K samples, raw (lon, lat) normalized, 10 seeds
- **Activations**: ReLU, Spline, SIREN
- **Question**: Can splines beat ReLU without SH pre-encoding?
- **Baseline**: NB21 showed SH+Spline +0.36% over SH+ReLU

### Experiment 2: Global Population (Raw Coords, Multi-Seed)
- **Config**: 15K samples, raw coords, 10 seeds
- **Question**: Does removal of SH change population task results?
- **Baseline**: NB19b showed SH+ReLU beats SH+Spline by 0.64%

### Experiment 3: Sample Size Sensitivity (Raw Coords)
- **Samples**: 5K, 10K, 20K, 50K
- **Question**: At what N do raw+learned results stabilize?
- **Compare to**: NB21 Exp 2 (with SH)

### Experiment 4: Regional Multi-Seed (Raw Coords)
- **Regions**: Himalayas (mountain), Sahara (flat)
- **Samples**: 10K, 20K per region
- **Question**: Do terrain effects appear without SH?
- **Compare to**: NB21 Exp 3 (with SH)

### Experiment 5: Extended Training (Raw Coords)
- **Epochs**: 200 vs 100
- **Question**: Convergence validation for raw coordinates

---

## Success Criteria

**Scenario A: Raw+Spline wins decisively**
- Raw+Spline advantage > 2%, CV < 20%, 95% CI excludes zero
- **Interpretation**: SH was masking spline benefits
- **Action**: Recommend raw+spline for applications, investigate why SH masks

**Scenario B: Raw+ReLU still wins**
- Raw+ReLU ≥ Raw+Spline consistently
- **Interpretation**: These tasks genuinely don't benefit from learned activations
- **Action**: Write up "When simplicity bias is sufficient"

**Scenario C: Raw << SH (for both acts)**
- SH improves both ReLU and Spline by >5%
- **Interpretation**: SH encoding is essential, justifies parameter cost
- **Action**: Stick with SH+ReLU as baseline

---

## Computational Budget

**Exp 1**: 10 seeds × 3 acts × 90s = ~45 min  
**Exp 2**: 10 seeds × 3 acts × 90s = ~45 min  
**Exp 3**: 4 sizes × 10 seeds × 2 acts × 60-120s = ~3 hours  
**Exp 4**: 2 regions × 2 sizes × 10 seeds × 2 acts × 100s = ~2 hours  
**Exp 5**: 5 seeds × 2 acts × 180s = ~30 min

**Total**: ~7 hours (can run in parallel with NB21/21b analysis)

---

## Key Differences from NB21/21b

| Aspect | NB21/21b | NB21c |
|--------|----------|-------|
| **Input encoding** | SH(L=10) = 121 dims | Raw (lon, lat) = 2 dims |
| **Hypothesis** | Test SH+acts reproducibility | Test if SH masks learned act gains |
| **Activations** | ReLU, Spline, SIREN | ReLU, Spline, SIREN |
| **Tasks** | Elevation, Population | Same |
| **Seeds** | 10 per config | Same |
| **Compare to** | NB19/19b single seed | NB21/21b with SH |

---

## Why Skip RFF?

- NB16 showed RFF works with raw coords (R² ~0.73)
- BUT: RFF+SH catastrophically fails (-7.96%)
- Focus on ReLU (baseline) and Spline (best learned act) for cleaner comparison
- Can add RFF in follow-up if spline shows promise

---
## Setup

In [1]:
# Environment setup
import os
import sys

if 'COLAB_GPU' in os.environ:
    !rm -rf sample_data .config satclip 2>/dev/null
    !git clone https://github.com/1hamzaiqbal/satclip.git
    !pip install lightning torchgeo huggingface_hub rasterio --quiet
    sys.path.append('./satclip/satclip')
else:
    sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'satclip'))

Cloning into 'satclip'...
remote: Enumerating objects: 635, done.
remote: Counting objects: 100% (314/314), done.
remote: Compressing objects: 100% (150/150), done.
remote: Total 635 (delta 226), reused 217 (delta 164), pack-reused 321 (from 2)
Receiving objects: 100% (635/635), 82.89 MiB | 15.55 MiB/s, done.
Resolving deltas: 100% (329/329), done.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.9/44.9 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.0/846.0 kB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 650.7/650.7 kB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.9/243.9 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 53.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 859.3/859.3 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.5/849.5 kB 61.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 11.7 MB/s eta 0:00:00
   ━━━

In [2]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import warnings
from scipy import stats
import rasterio
import zipfile
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import r2_score

import xarray as xr

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cuda


---
## Model Definitions (Raw Coords Only)

In [3]:
class SplineActivation(nn.Module):
    def __init__(self, n_knots=15, input_range=(-3.0, 3.0), init='relu'):
        super().__init__()
        self.n_knots = n_knots
        self.input_range = input_range
        knot_x = torch.linspace(input_range[0], input_range[1], n_knots)
        self.register_buffer('knot_x', knot_x)
        if init == 'relu':
            knot_y = torch.relu(knot_x)
        elif init == 'linear':
            knot_y = knot_x.clone()
        else:
            knot_y = torch.randn(n_knots) * 0.1
        self.knot_y = nn.Parameter(knot_y)

    def forward(self, x):
        x_clamped = torch.clamp(x, self.input_range[0], self.input_range[1])
        x_norm = (x_clamped - self.knot_x[0]) / (self.knot_x[-1] - self.knot_x[0])
        x_idx = x_norm * (self.n_knots - 1)
        idx_low = torch.floor(x_idx).long()
        idx_high = torch.clamp(idx_low + 1, max=self.n_knots - 1)
        idx_low = torch.clamp(idx_low, max=self.n_knots - 1)
        weight = x_idx - idx_low.float()
        y_low = self.knot_y[idx_low]
        y_high = self.knot_y[idx_high]
        return y_low + weight * (y_high - y_low)


class SirenLayer(nn.Module):
    def __init__(self, dim_in, dim_out, w0=1.0, is_first=False):
        super().__init__()
        self.dim_in = dim_in
        self.w0 = w0
        self.is_first = is_first
        self.linear = nn.Linear(dim_in, dim_out)
        self._init_weights()

    def _init_weights(self):
        if self.is_first:
            bound = 1.0 / self.dim_in
        else:
            bound = np.sqrt(6.0 / self.dim_in) / self.w0
        self.linear.weight.data.uniform_(-bound, bound)
        if self.linear.bias is not None:
            self.linear.bias.data.uniform_(-bound, bound)

    def forward(self, x):
        return torch.sin(self.w0 * self.linear(x))


class RawCoordinateEncoder(nn.Module):
    """
    Encoder for RAW COORDINATES ONLY (no SH, no RFF).
    Input: (lon, lat) normalized to [-1, 1] × [-1, 1]
    """
    def __init__(self, activation_type='relu', activation_kwargs=None,
                 n_layers=3, hidden_dim=256, output_dim=256):
        super().__init__()
        self.activation_type = activation_type

        input_dim = 2  # Raw (lon, lat)

        if activation_kwargs is None:
            activation_kwargs = {}

        dims = [input_dim] + [hidden_dim] * n_layers + [output_dim]

        if activation_type == 'siren':
            self.layers = nn.ModuleList()
            for i in range(len(dims) - 1):
                is_first = (i == 0)
                w0 = 30.0 if is_first else 1.0
                if i < len(dims) - 2:
                    self.layers.append(SirenLayer(dims[i], dims[i+1], w0=w0, is_first=is_first))
                else:
                    linear = nn.Linear(dims[i], dims[i+1])
                    bound = np.sqrt(6.0 / dims[i]) / 1.0
                    linear.weight.data.uniform_(-bound, bound)
                    if linear.bias is not None:
                        linear.bias.data.uniform_(-bound, bound)
                    self.layers.append(linear)
            self.activations = None

        elif activation_type == 'spline':
            self.linears = nn.ModuleList([
                nn.Linear(dims[i], dims[i+1])
                for i in range(len(dims) - 1)
            ])
            self.activations = nn.ModuleList([
                SplineActivation(**activation_kwargs)
                for _ in range(n_layers)
            ])
            for linear in self.linears:
                nn.init.kaiming_normal_(linear.weight)
                nn.init.zeros_(linear.bias)

        elif activation_type == 'relu':
            layers = []
            for i in range(len(dims) - 1):
                layers.append(nn.Linear(dims[i], dims[i+1]))
                if i < len(dims) - 2:
                    layers.append(nn.ReLU())
            self.net = nn.Sequential(*layers)
            for m in self.modules():
                if isinstance(m, nn.Linear):
                    nn.init.kaiming_normal_(m.weight)
                    nn.init.zeros_(m.bias)
        else:
            raise ValueError(f"Unknown activation_type: {activation_type}")

    def forward(self, coords):
        """
        coords: (batch, 2) with lon in [-180, 180], lat in [-90, 90]
        Normalize to [-1, 1] × [-1, 1]
        """
        x = coords / torch.tensor([180., 90.], device=coords.device)

        if self.activation_type == 'siren':
            for layer in self.layers:
                x = layer(x)
        elif self.activation_type == 'spline':
            for i, (linear, act) in enumerate(zip(self.linears[:-1], self.activations)):
                x = act(linear(x))
            x = self.linears[-1](x)
        else:
            x = self.net(x)

        return x


class RegressionPredictor(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        self.head = nn.Sequential(
            nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, 1)
        )

    def forward(self, coords):
        return self.head(self.encoder(coords)).squeeze(-1)


print("✅ Model classes loaded (RAW COORDINATES ONLY)")

✅ Model classes loaded (RAW COORDINATES ONLY)


---
## Data Loading & Sampling Utilities (Same as NB21)

In [4]:
def sample_global_blocked(data, lons, lats, n_samples=15000, grid_size=5.0, test_ratio=0.3, seed=42):
    """
    Sample globally with spatial blocking.
    """
    np.random.seed(seed)

    # Valid data mask
    valid = data > -1e30

    # Create meshgrid
    lon_grid, lat_grid = np.meshgrid(lons, lats)

    # Flatten
    valid_lons = lon_grid[valid]
    valid_lats = lat_grid[valid]
    valid_vals = data[valid]

    # Sample subset
    n_valid = len(valid_vals)
    if n_valid > n_samples:
        sample_idx = np.random.choice(n_valid, n_samples, replace=False)
        sample_lons = valid_lons[sample_idx]
        sample_lats = valid_lats[sample_idx]
        sample_vals = valid_vals[sample_idx]
    else:
        sample_lons = valid_lons
        sample_lats = valid_lats
        sample_vals = valid_vals

    # Spatial blocking
    n_lon_cells = int(360 / grid_size)
    n_lat_cells = int(180 / grid_size)
    n_cells = n_lon_cells * n_lat_cells

    test_cells = set(np.random.choice(n_cells, int(n_cells * test_ratio), replace=False))

    train_mask = []
    for lon, lat in zip(sample_lons, sample_lats):
        lon_cell = int((lon + 180) / grid_size)
        lat_cell = int((lat + 90) / grid_size)
        lon_cell = min(lon_cell, n_lon_cells - 1)
        lat_cell = min(lat_cell, n_lat_cells - 1)
        cell = lat_cell * n_lon_cells + lon_cell
        train_mask.append(cell not in test_cells)

    train_mask = np.array(train_mask)
    coords = np.stack([sample_lons, sample_lats], axis=1)

    return coords[train_mask], sample_vals[train_mask], coords[~train_mask], sample_vals[~train_mask]


def sample_regional_blocked(data, lons, lats, region_bounds, n_samples=5000,
                           grid_size=5.0, test_ratio=0.3, seed=42):
    """
    Sample from a regional subset with spatial blocking.
    region_bounds: dict with 'lat_min', 'lat_max', 'lon_min', 'lon_max'
    """
    np.random.seed(seed)

    # Find indices for region
    lat_mask = (lats >= region_bounds['lat_min']) & (lats <= region_bounds['lat_max'])
    lon_mask = (lons >= region_bounds['lon_min']) & (lons <= region_bounds['lon_max'])

    lat_idx = np.where(lat_mask)[0]
    lon_idx = np.where(lon_mask)[0]

    regional_data = data[np.ix_(lat_idx, lon_idx)]
    regional_lats = lats[lat_idx]
    regional_lons = lons[lon_idx]

    lon_grid, lat_grid = np.meshgrid(regional_lons, regional_lats)
    valid = regional_data > -1e30

    valid_lons = lon_grid[valid]
    valid_lats = lat_grid[valid]
    valid_vals = regional_data[valid]

    n_valid = len(valid_vals)
    if n_valid > n_samples:
        sample_idx = np.random.choice(n_valid, n_samples, replace=False)
        sample_lons = valid_lons[sample_idx]
        sample_lats = valid_lats[sample_idx]
        sample_vals = valid_vals[sample_idx]
    else:
        sample_lons = valid_lons
        sample_lats = valid_lats
        sample_vals = valid_vals

    region_width = region_bounds['lon_max'] - region_bounds['lon_min']
    region_height = region_bounds['lat_max'] - region_bounds['lat_min']

    n_lon_cells = max(1, int(region_width / grid_size))
    n_lat_cells = max(1, int(region_height / grid_size))
    n_cells = n_lon_cells * n_lat_cells

    test_cells = set(np.random.choice(n_cells, max(1, int(n_cells * test_ratio)), replace=False))

    train_mask = []
    for lon, lat in zip(sample_lons, sample_lats):
        lon_cell = int((lon - region_bounds['lon_min']) / grid_size)
        lat_cell = int((lat - region_bounds['lat_min']) / grid_size)
        lon_cell = min(lon_cell, n_lon_cells - 1)
        lat_cell = min(lat_cell, n_lat_cells - 1)
        cell = lat_cell * n_lon_cells + lon_cell
        train_mask.append(cell not in test_cells)

    train_mask = np.array(train_mask)
    coords = np.stack([sample_lons, sample_lats], axis=1)

    return coords[train_mask], sample_vals[train_mask], coords[~train_mask], sample_vals[~train_mask]


print("✅ Sampling utilities loaded")

✅ Sampling utilities loaded


---
## Training Functions

In [5]:
def train_model(name, encoder, coords_train, vals_train, coords_test, vals_test,
               task_type='elevation', epochs=100, batch_size=256, lr=1e-3,
               verbose=False, track_epochs=False):
    """
    Train model with raw coordinates.
    task_type: 'elevation' or 'population'
    """
    model = RegressionPredictor(encoder).to(device)
    opt = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    # Target transform
    if task_type == 'elevation':
        # Normalize + shift + log
        train_mean, train_std = vals_train.mean(), vals_train.std()
        vals_train_norm = (vals_train - train_mean) / train_std
        vals_test_norm = (vals_test - train_mean) / train_std

        shift = -vals_train_norm.min() + 1
        train_y = np.log1p(vals_train_norm + shift)
        test_y = np.log1p(vals_test_norm + shift)
    else:  # population
        train_y = np.log1p(vals_train)
        test_y = np.log1p(vals_test)

    train_X = torch.tensor(coords_train, dtype=torch.float32)
    train_y = torch.tensor(train_y, dtype=torch.float32)
    test_X = torch.tensor(coords_test, dtype=torch.float32).to(device)
    test_y = torch.tensor(test_y, dtype=torch.float32)

    loader = DataLoader(TensorDataset(train_X, train_y),
                       batch_size=batch_size, shuffle=True)

    best_r2 = -float('inf')
    r2_history = [] if track_epochs else None
    start = time.time()

    for epoch in range(epochs):
        model.train()
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            opt.zero_grad()
            loss = loss_fn(model(X), y)
            loss.backward()
            opt.step()

        # Evaluate
        if (epoch + 1) % 10 == 0 or epoch == epochs - 1 or track_epochs:
            model.eval()
            with torch.no_grad():
                pred = model(test_X).cpu().numpy()
            r2 = r2_score(test_y.numpy(), pred)
            best_r2 = max(best_r2, r2)

            if track_epochs:
                r2_history.append(r2)

            if verbose and (epoch + 1) % 20 == 0:
                print(f"  Epoch {epoch+1}/{epochs}: R² = {r2:.4f}")

    train_time = time.time() - start
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    result = {
        'model': name,
        'r2': best_r2,
        'params': n_params,
        'time': train_time,
    }

    if track_epochs:
        result['r2_history'] = r2_history

    return result


print("✅ Training functions loaded")

✅ Training functions loaded


---
## Statistics Utilities (Same as NB21)

In [6]:
def compute_stats(values):
    """Compute comprehensive statistics."""
    values = np.array(values)
    n = len(values)
    mean = values.mean()
    std = values.std(ddof=1) if n > 1 else 0.0
    stderr = std / np.sqrt(n) if n > 1 else 0.0

    if n > 1:
        ci = stats.t.interval(0.95, n-1, loc=mean, scale=stderr)
    else:
        ci = (mean, mean)

    return {
        'n': n,
        'mean': mean,
        'std': std,
        'stderr': stderr,
        'min': values.min(),
        'max': values.max(),
        'ci_low': ci[0],
        'ci_high': ci[1],
        'cv': (std / mean * 100) if mean != 0 else 0.0,
    }


def print_stats(label, stats_dict):
    """Pretty print statistics."""
    print(f"{label}:")
    print(f"  Mean ± Std:  {stats_dict['mean']:.4f} ± {stats_dict['std']:.4f}")
    print(f"  Range:       [{stats_dict['min']:.4f}, {stats_dict['max']:.4f}]")
    print(f"  95% CI:      [{stats_dict['ci_low']:.4f}, {stats_dict['ci_high']:.4f}]")
    print(f"  CV:          {stats_dict['cv']:.2f}%")
    print(f"  N:           {stats_dict['n']}")


def compare_activations(relu_r2s, spline_r2s, label="SPLINE vs RELU"):
    """Compare ReLU vs Spline with statistical tests."""
    relu_r2s = np.array(relu_r2s)
    spline_r2s = np.array(spline_r2s)

    if len(relu_r2s) == len(spline_r2s) and len(relu_r2s) > 1:
        t_stat, p_value = stats.ttest_rel(spline_r2s, relu_r2s)
    else:
        t_stat, p_value = np.nan, np.nan

    advantages = 100 * (spline_r2s - relu_r2s) / relu_r2s
    adv_stats = compute_stats(advantages)

    print("\n" + "="*60)
    print(label)
    print("="*60)

    relu_stats = compute_stats(relu_r2s)
    spline_stats = compute_stats(spline_r2s)

    print("\nReLU:")
    print(f"  Mean ± Std:  {relu_stats['mean']:.4f} ± {relu_stats['std']:.4f}")
    print(f"  95% CI:      [{relu_stats['ci_low']:.4f}, {relu_stats['ci_high']:.4f}]")

    print("\nSpline:")
    print(f"  Mean ± Std:  {spline_stats['mean']:.4f} ± {spline_stats['std']:.4f}")
    print(f"  95% CI:      [{spline_stats['ci_low']:.4f}, {spline_stats['ci_high']:.4f}]")

    print("\nSpline Advantage (%):")
    print(f"  Mean ± Std:  {adv_stats['mean']:+.2f}% ± {adv_stats['std']:.2f}%")
    print(f"  Range:       [{adv_stats['min']:+.2f}%, {adv_stats['max']:+.2f}%]")
    print(f"  95% CI:      [{adv_stats['ci_low']:+.2f}%, {adv_stats['ci_high']:+.2f}%]")

    if not np.isnan(p_value):
        print(f"\nPaired t-test: t={t_stat:.3f}, p={p_value:.4f}")
        if p_value < 0.05:
            if adv_stats['mean'] > 0:
                print("  ✅ SIGNIFICANT: Spline wins (p < 0.05)")
            else:
                print("  ✅ SIGNIFICANT: ReLU wins (p < 0.05)")
        else:
            print("  ❌ NOT SIGNIFICANT: No clear winner (p ≥ 0.05)")

    if adv_stats['ci_low'] > 1.0:
        print("\n  🎯 PRACTICAL SIGNIFICANCE: 95% CI excludes zero, advantage > 1%")
    elif adv_stats['ci_high'] < -1.0:
        print("\n  🎯 PRACTICAL SIGNIFICANCE: 95% CI excludes zero, disadvantage > 1%")
    else:
        print("\n  ⚠️  NO PRACTICAL SIGNIFICANCE: 95% CI includes small effects")

    print("="*60)

    return adv_stats


print("✅ Statistics utilities loaded")

✅ Statistics utilities loaded


---
## Load Elevation Data

In [7]:
print("="*70)
print("LOADING ETOPO ELEVATION DATA")
print("="*70)

if 'COLAB_GPU' in os.environ:
    !wget -q -O etopo_60s.nc "https://www.ngdc.noaa.gov/thredds/fileServer/global/ETOPO2022/60s/60s_surface_elev_netcdf/ETOPO_2022_v1_60s_N90W180_surface.nc"
    data_path = 'etopo_60s.nc'
else:
    data_path = 'etopo_60s.nc'

ds = xr.open_dataset(data_path)
elevation = ds['z'].values
elev_lats = ds['lat'].values
elev_lons = ds['lon'].values

print(f"✅ Elevation: {elevation.shape}")
print(f"   Range: [{elevation.min():.2f}, {elevation.max():.2f}] m")
print("="*70)

LOADING ETOPO ELEVATION DATA
✅ Elevation: (10800, 21600)
   Range: [-10752.08, 8157.36] m


---
## Load Population Data

In [8]:
print("="*70)
print("LOADING GPW POPULATION DATA")
print("="*70)

if 'COLAB_GPU' in os.environ:
    from google.colab import drive
    drive.mount('/content/drive')

    GPW_DIR = './gpw_data'
    os.makedirs(GPW_DIR, exist_ok=True)

    SOURCE_ZIP_PATH = '/content/drive/MyDrive/grad/learned_activations/dataverse_files.zip'

    print("Extracting GPW data...")
    with zipfile.ZipFile(SOURCE_ZIP_PATH, 'r') as z:
        z.extractall(GPW_DIR)

    # Extract 15_min TIF
    zip_name = "gpw-v4-population-density-rev11_2020_15_min_tif.zip"
    zip_path = os.path.join(GPW_DIR, zip_name)
    if os.path.exists(zip_path):
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(GPW_DIR)

    # Load TIF
    tif_filename = 'gpw_v4_population_density_rev11_2020_15_min.tif'
    tif_path = os.path.join(GPW_DIR, tif_filename)

    with rasterio.open(tif_path) as src:
        population = src.read(1)
        transform = src.transform
        height, width = population.shape

        pop_lons = np.array([transform * (i, 0) for i in range(width)])[:, 0]
        pop_lats = np.array([transform * (0, j) for j in range(height)])[:, 1]

        nodata = src.nodata
        if nodata is not None:
            population = np.where(population == nodata, -9999, population)

        population = np.where(population <= 0, 1e-6, population)
else:
    GPW_DIR = './gpw_data'
    print("Using local GPW data")

    # Assume already extracted
    tif_filename = 'gpw_v4_population_density_rev11_2020_15_min.tif'
    tif_path = os.path.join(GPW_DIR, tif_filename)

    with rasterio.open(tif_path) as src:
        population = src.read(1)
        transform = src.transform
        height, width = population.shape

        pop_lons = np.array([transform * (i, 0) for i in range(width)])[:, 0]
        pop_lats = np.array([transform * (0, j) for j in range(height)])[:, 1]

        nodata = src.nodata
        if nodata is not None:
            population = np.where(population == nodata, -9999, population)

        population = np.where(population <= 0, 1e-6, population)

print(f"✅ Population: {population.shape}")
print(f"   Range: [{population[population > 0].min():.2e}, {population.max():.2e}] people/km²")
print("="*70)

LOADING GPW POPULATION DATA
Mounted at /content/drive
Extracting GPW data...
✅ Population: (720, 1440)
   Range: [1.92e-09, 2.98e+04] people/km²


---
## Experiment 1: Global Elevation (Raw Coords, Multi-Seed)

**Question**: Can splines beat ReLU without SH encoding?

**Compare to NB21**: SH+Spline showed +0.36% over SH+ReLU

In [9]:
print("="*80)
print("EXPERIMENT 1: GLOBAL ELEVATION (Raw Coordinates, Multi-Seed)")
print("="*80)

SEEDS = range(42, 52)  # 10 seeds
ACTIVATIONS = ['relu', 'spline', 'siren']
N_SAMPLES = 15000
EPOCHS = 100

results_exp1_elev = []

for seed in SEEDS:
    print(f"\n{'='*80}")
    print(f"Seed {seed} ({seed-41}/10)")
    print("="*80)

    coords_train, vals_train, coords_test, vals_test = sample_global_blocked(
        elevation, elev_lons, elev_lats, n_samples=N_SAMPLES, seed=seed
    )
    print(f"Samples: {len(coords_train)} train, {len(coords_test)} test")

    for act in ACTIVATIONS:
        print(f"\n  {act.upper()}...", end=" ")

        kwargs = {'n_knots': 15, 'init': 'relu'} if act == 'spline' else None

        enc = RawCoordinateEncoder(
            activation_type=act,
            activation_kwargs=kwargs
        )

        res = train_model(
            f'raw_elev_seed{seed}_{act}',
            enc,
            coords_train, vals_train,
            coords_test, vals_test,
            task_type='elevation',
            epochs=EPOCHS,
            verbose=False
        )

        res['seed'] = seed
        res['activation'] = act
        res['task'] = 'elevation'
        res['encoding'] = 'raw'

        results_exp1_elev.append(res)
        print(f"R²={res['r2']:.4f}, Time={res['time']:.1f}s")

df_exp1_elev = pd.DataFrame(results_exp1_elev)

print("\n" + "="*80)
print("EXPERIMENT 1 (ELEVATION): RESULTS")
print("="*80)
print(df_exp1_elev[['seed', 'activation', 'r2', 'time']].to_string(index=False))
print("="*80)

EXPERIMENT 1: GLOBAL ELEVATION (Raw Coordinates, Multi-Seed)

Seed 42 (1/10)
Samples: 10479 train, 4521 test

  RELU... R²=0.8216, Time=16.6s

  SPLINE... R²=0.8724, Time=41.6s

  SIREN... R²=0.8861, Time=16.2s

Seed 43 (2/10)
Samples: 10559 train, 4441 test

  RELU... R²=0.8187, Time=15.8s

  SPLINE... R²=0.8579, Time=45.0s

  SIREN... R²=0.8621, Time=16.7s

Seed 44 (3/10)
Samples: 10514 train, 4486 test

  RELU... R²=0.8184, Time=15.8s

  SPLINE... R²=0.8858, Time=40.9s

  SIREN... R²=0.8803, Time=16.7s

Seed 45 (4/10)
Samples: 10528 train, 4472 test

  RELU... R²=0.8283, Time=15.8s

  SPLINE... R²=0.8625, Time=45.4s

  SIREN... R²=0.8791, Time=16.6s

Seed 46 (5/10)
Samples: 10522 train, 4478 test

  RELU... R²=0.8308, Time=15.9s

  SPLINE... R²=0.8626, Time=45.8s

  SIREN... R²=0.8917, Time=16.7s

Seed 47 (6/10)
Samples: 10530 train, 4470 test

  RELU... R²=0.8056, Time=15.9s

  SPLINE... R²=0.8717, Time=44.9s

  SIREN... R²=0.8728, Time=16.7s

Seed 48 (7/10)
Samples: 10488 train, 4

In [10]:
# Statistical analysis
print("\n" + "="*80)
print("EXPERIMENT 1 (ELEVATION): STATISTICAL ANALYSIS")
print("="*80)

for act in ACTIVATIONS:
    act_data = df_exp1_elev[df_exp1_elev['activation'] == act]
    r2_values = act_data['r2'].values
    stats_dict = compute_stats(r2_values)
    print(f"\n{act.upper()}:")
    print_stats(f"  R² Statistics", stats_dict)

relu_r2s = df_exp1_elev[df_exp1_elev['activation'] == 'relu']['r2'].values
spline_r2s = df_exp1_elev[df_exp1_elev['activation'] == 'spline']['r2'].values

adv_stats_elev = compare_activations(relu_r2s, spline_r2s, "RAW+SPLINE vs RAW+RELU (ELEVATION)")

print("\n" + "="*80)
print("COMPARISON TO NB21 (with SH encoding)")
print("="*80)
print("\n⚠️  Fill in NB21 results after it completes:")
print("\nNB21 (SH+acts):")
print("  SH+ReLU:   R² = [FROM NB21 EXP 1]")
print("  SH+Spline: R² = [FROM NB21 EXP 1]")
print("  Advantage: [FROM NB21]")
print("\nNB21c (Raw+acts):")
print(f"  Raw+ReLU:   R² = {relu_r2s.mean():.4f} ± {relu_r2s.std():.4f}")
print(f"  Raw+Spline: R² = {spline_r2s.mean():.4f} ± {spline_r2s.std():.4f}")
print(f"  Advantage: {adv_stats_elev['mean']:+.2f}% ± {adv_stats_elev['std']:.2f}%")

print("\n" + "="*80)
print("KEY QUESTION: Does SH mask spline gains?")
print("="*80)
print("\nIF Raw+Spline advantage > SH+Spline advantage:")
print("  → YES, SH pre-encoding masks learned activation benefits")
print("\nIF SH+Spline advantage > Raw+Spline advantage:")
print("  → NO, SH and splines are complementary")
print("="*80)


EXPERIMENT 1 (ELEVATION): STATISTICAL ANALYSIS

RELU:
  R² Statistics:
  Mean ± Std:  0.8222 ± 0.0127
  Range:       [0.8056, 0.8465]
  95% CI:      [0.8131, 0.8313]
  CV:          1.55%
  N:           10

SPLINE:
  R² Statistics:
  Mean ± Std:  0.8721 ± 0.0105
  Range:       [0.8579, 0.8877]
  95% CI:      [0.8646, 0.8796]
  CV:          1.20%
  N:           10

SIREN:
  R² Statistics:
  Mean ± Std:  0.8803 ± 0.0093
  Range:       [0.8621, 0.8917]
  95% CI:      [0.8737, 0.8870]
  CV:          1.06%
  N:           10

RAW+SPLINE vs RAW+RELU (ELEVATION)

ReLU:
  Mean ± Std:  0.8222 ± 0.0127
  95% CI:      [0.8131, 0.8313]

Spline:
  Mean ± Std:  0.8721 ± 0.0105
  95% CI:      [0.8646, 0.8796]

Spline Advantage (%):
  Mean ± Std:  +6.09% ± 2.10%
  Range:       [+3.83%, +9.32%]
  95% CI:      [+4.58%, +7.59%]

Paired t-test: t=9.476, p=0.0000
  ✅ SIGNIFICANT: Spline wins (p < 0.05)

  🎯 PRACTICAL SIGNIFICANCE: 95% CI excludes zero, advantage > 1%

COMPARISON TO NB21 (with SH encoding)



---
## Experiment 2: Global Population (Raw Coords, Multi-Seed)

**Question**: Does raw encoding change population results?

**Compare to NB21b**: SH+ReLU beat SH+Spline by 0.64%

In [11]:
print("="*80)
print("EXPERIMENT 2: GLOBAL POPULATION (Raw Coordinates, Multi-Seed)")
print("="*80)

results_exp2_pop = []

for seed in SEEDS:
    print(f"\n{'='*80}")
    print(f"Seed {seed} ({seed-41}/10)")
    print("="*80)

    coords_train, vals_train, coords_test, vals_test = sample_global_blocked(
        population, pop_lons, pop_lats, n_samples=N_SAMPLES, seed=seed
    )
    print(f"Samples: {len(coords_train)} train, {len(coords_test)} test")

    for act in ACTIVATIONS:
        print(f"\n  {act.upper()}...", end=" ")

        kwargs = {'n_knots': 15, 'init': 'relu'} if act == 'spline' else None

        enc = RawCoordinateEncoder(
            activation_type=act,
            activation_kwargs=kwargs
        )

        res = train_model(
            f'raw_pop_seed{seed}_{act}',
            enc,
            coords_train, vals_train,
            coords_test, vals_test,
            task_type='population',
            epochs=EPOCHS,
            verbose=False
        )

        res['seed'] = seed
        res['activation'] = act
        res['task'] = 'population'
        res['encoding'] = 'raw'

        results_exp2_pop.append(res)
        print(f"R²={res['r2']:.4f}, Time={res['time']:.1f}s")

df_exp2_pop = pd.DataFrame(results_exp2_pop)

print("\n" + "="*80)
print("EXPERIMENT 2 (POPULATION): RESULTS")
print("="*80)
print(df_exp2_pop[['seed', 'activation', 'r2', 'time']].to_string(index=False))
print("="*80)

EXPERIMENT 2: GLOBAL POPULATION (Raw Coordinates, Multi-Seed)

Seed 42 (1/10)
Samples: 10512 train, 4488 test

  RELU... R²=0.4778, Time=15.8s

  SPLINE... R²=0.5351, Time=40.6s

  SIREN... R²=0.5496, Time=16.7s

Seed 43 (2/10)
Samples: 10486 train, 4514 test

  RELU... R²=0.6041, Time=15.9s

  SPLINE... R²=0.6296, Time=39.1s

  SIREN... R²=0.5947, Time=16.4s

Seed 44 (3/10)
Samples: 10592 train, 4408 test

  RELU... R²=0.4942, Time=16.1s

  SPLINE... R²=0.5351, Time=39.0s

  SIREN... R²=0.5250, Time=16.7s

Seed 45 (4/10)
Samples: 10519 train, 4481 test

  RELU... R²=0.5494, Time=15.8s

  SPLINE... R²=0.6070, Time=43.3s

  SIREN... R²=0.5943, Time=17.1s

Seed 46 (5/10)
Samples: 10513 train, 4487 test

  RELU... R²=0.5043, Time=16.2s

  SPLINE... R²=0.5613, Time=40.3s

  SIREN... R²=0.5790, Time=16.7s

Seed 47 (6/10)
Samples: 10605 train, 4395 test

  RELU... R²=0.6085, Time=15.9s

  SPLINE... R²=0.6308, Time=38.6s

  SIREN... R²=0.6027, Time=16.8s

Seed 48 (7/10)
Samples: 10525 train, 

In [12]:
# Statistical analysis
print("\n" + "="*80)
print("EXPERIMENT 2 (POPULATION): STATISTICAL ANALYSIS")
print("="*80)

for act in ACTIVATIONS:
    act_data = df_exp2_pop[df_exp2_pop['activation'] == act]
    r2_values = act_data['r2'].values
    stats_dict = compute_stats(r2_values)
    print(f"\n{act.upper()}:")
    print_stats(f"  R² Statistics", stats_dict)

relu_r2s_pop = df_exp2_pop[df_exp2_pop['activation'] == 'relu']['r2'].values
spline_r2s_pop = df_exp2_pop[df_exp2_pop['activation'] == 'spline']['r2'].values

adv_stats_pop = compare_activations(relu_r2s_pop, spline_r2s_pop, "RAW+SPLINE vs RAW+RELU (POPULATION)")

print("\n" + "="*80)
print("COMPARISON TO NB21b (with SH encoding)")
print("="*80)
print("\n⚠️  Fill in NB21b results after it completes:")
print("\nNB21b (SH+acts):")
print("  SH+ReLU:   R² = [FROM NB21b EXP 1]")
print("  SH+Spline: R² = [FROM NB21b EXP 1]")
print("  Advantage: [FROM NB21b] (NB19b showed -0.64%)")
print("\nNB21c (Raw+acts):")
print(f"  Raw+ReLU:   R² = {relu_r2s_pop.mean():.4f} ± {relu_r2s_pop.std():.4f}")
print(f"  Raw+Spline: R² = {spline_r2s_pop.mean():.4f} ± {spline_r2s_pop.std():.4f}")
print(f"  Advantage: {adv_stats_pop['mean']:+.2f}% ± {adv_stats_pop['std']:.2f}%")
print("="*80)


EXPERIMENT 2 (POPULATION): STATISTICAL ANALYSIS

RELU:
  R² Statistics:
  Mean ± Std:  0.5375 ± 0.0431
  Range:       [0.4778, 0.6085]
  95% CI:      [0.5066, 0.5683]
  CV:          8.03%
  N:           10

SPLINE:
  R² Statistics:
  Mean ± Std:  0.5832 ± 0.0349
  Range:       [0.5351, 0.6308]
  95% CI:      [0.5582, 0.6082]
  CV:          5.99%
  N:           10

SIREN:
  R² Statistics:
  Mean ± Std:  0.5768 ± 0.0253
  Range:       [0.5250, 0.6027]
  95% CI:      [0.5587, 0.5950]
  CV:          4.39%
  N:           10

RAW+SPLINE vs RAW+RELU (POPULATION)

ReLU:
  Mean ± Std:  0.5375 ± 0.0431
  95% CI:      [0.5066, 0.5683]

Spline:
  Mean ± Std:  0.5832 ± 0.0349
  95% CI:      [0.5582, 0.6082]

Spline Advantage (%):
  Mean ± Std:  +8.68% ± 2.92%
  Range:       [+3.67%, +11.99%]
  95% CI:      [+6.59%, +10.77%]

Paired t-test: t=10.540, p=0.0000
  ✅ SIGNIFICANT: Spline wins (p < 0.05)

  🎯 PRACTICAL SIGNIFICANCE: 95% CI excludes zero, advantage > 1%

COMPARISON TO NB21b (with SH encod

---
## Experiment 3: Sample Size Sensitivity (Raw Coords)

**Question**: At what N do raw+learned results stabilize?

In [13]:
print("="*80)
print("EXPERIMENT 3: SAMPLE SIZE SENSITIVITY (Raw Coordinates)")
print("="*80)

SAMPLE_SIZES = [5000, 10000, 20000, 50000]
SEEDS_EXP3 = range(42, 52)  # 10 seeds
ACTIVATIONS_EXP3 = ['relu', 'spline']  # Focus on main comparison

results_exp3 = []

for n_samples in SAMPLE_SIZES:
    print(f"\n{'='*80}")
    print(f"Sample Size: {n_samples:,}")
    print("="*80)

    for seed in SEEDS_EXP3:
        print(f"  Seed {seed}: ", end="")

        # Use elevation for this experiment
        coords_train, vals_train, coords_test, vals_test = sample_global_blocked(
            elevation, elev_lons, elev_lats, n_samples=n_samples, seed=seed
        )

        for act in ACTIVATIONS_EXP3:
            kwargs = {'n_knots': 15, 'init': 'relu'} if act == 'spline' else None

            enc = RawCoordinateEncoder(
                activation_type=act,
                activation_kwargs=kwargs
            )

            res = train_model(
                f'raw_n{n_samples}_seed{seed}_{act}',
                enc,
                coords_train, vals_train,
                coords_test, vals_test,
                task_type='elevation',
                epochs=EPOCHS,
                verbose=False
            )

            res['seed'] = seed
            res['activation'] = act
            res['n_samples'] = n_samples
            res['encoding'] = 'raw'

            results_exp3.append(res)
            print(f"{act}={res['r2']:.4f} ", end="")

        print()

df_exp3 = pd.DataFrame(results_exp3)

print("\n" + "="*80)
print("EXPERIMENT 3: VARIANCE vs SAMPLE SIZE")
print("="*80)

print("\n{:>10s} {:>15s} {:>15s} {:>10s} {:>10s}".format(
    "N Samples", "ReLU R²", "Spline R²", "Adv (%)", "Adv CV"
))
print("-" * 80)

for n_samples in SAMPLE_SIZES:
    subset = df_exp3[df_exp3['n_samples'] == n_samples]
    relu_r2s = subset[subset['activation'] == 'relu']['r2'].values
    spline_r2s = subset[subset['activation'] == 'spline']['r2'].values

    relu_stats = compute_stats(relu_r2s)
    spline_stats = compute_stats(spline_r2s)
    advantages = 100 * (spline_r2s - relu_r2s) / relu_r2s
    adv_stats = compute_stats(advantages)

    print("{:>10,d} {:>15s} {:>15s} {:>10.2f} {:>10.2f}".format(
        n_samples,
        f"{relu_stats['mean']:.4f}±{relu_stats['std']:.4f}",
        f"{spline_stats['mean']:.4f}±{spline_stats['std']:.4f}",
        adv_stats['mean'],
        adv_stats['cv']
    ))

print("\n⚠️  Compare to NB21 Exp 2 (with SH) to see if variance patterns differ")
print("="*80)

EXPERIMENT 3: SAMPLE SIZE SENSITIVITY (Raw Coordinates)

Sample Size: 5,000
  Seed 42: relu=0.7472 spline=0.8199 
  Seed 43: relu=0.7462 spline=0.8124 
  Seed 44: relu=0.7471 spline=0.8451 
  Seed 45: relu=0.7431 spline=0.8403 
  Seed 46: relu=0.7429 spline=0.8304 
  Seed 47: relu=0.7282 spline=0.8255 
  Seed 48: relu=0.7643 spline=0.8548 
  Seed 49: relu=0.7539 spline=0.8383 
  Seed 50: relu=0.7460 spline=0.8471 
  Seed 51: relu=0.7628 spline=0.8359 

Sample Size: 10,000
  Seed 42: relu=0.8046 spline=0.8487 
  Seed 43: relu=0.8124 spline=0.8306 
  Seed 44: relu=0.7912 spline=0.8542 
  Seed 45: relu=0.8316 spline=0.8561 
  Seed 46: relu=0.7925 spline=0.8646 
  Seed 47: relu=0.8049 spline=0.8654 
  Seed 48: relu=0.8243 spline=0.8712 
  Seed 49: relu=0.7916 spline=0.8645 
  Seed 50: relu=0.7992 spline=0.8649 
  Seed 51: relu=0.7916 spline=0.8289 

Sample Size: 20,000
  Seed 42: relu=0.8441 spline=0.8827 
  Seed 43: relu=0.8081 spline=0.8571 
  Seed 44: relu=0.8484 spline=0.8719 
  Seed 4

---
## Experiment 4: Regional Multi-Seed (Raw Coords)

**Question**: Do terrain effects appear without SH?

In [14]:
print("="*80)
print("EXPERIMENT 4: REGIONAL MULTI-SEED (Raw Coordinates)")
print("="*80)

REGIONS = {
    'asia_himalayas': {
        'lat_min': 25, 'lat_max': 40,
        'lon_min': 70, 'lon_max': 100,
        'terrain': 'mountainous',
    },
    'africa_sahara': {
        'lat_min': 15, 'lat_max': 30,
        'lon_min': -10, 'lon_max': 30,
        'terrain': 'flat',
    },
}

REGIONAL_SAMPLE_SIZES = [10000, 20000]
SEEDS_EXP4 = range(42, 52)  # 10 seeds
ACTIVATIONS_EXP4 = ['relu', 'spline']

results_exp4 = []

for region_name, region_bounds in REGIONS.items():
    print(f"\n{'='*80}")
    print(f"Region: {region_name.upper()} ({region_bounds['terrain']})")
    print("="*80)

    for n_samples in REGIONAL_SAMPLE_SIZES:
        print(f"\n  Sample Size: {n_samples:,}")

        for seed in SEEDS_EXP4:
            print(f"    Seed {seed}: ", end="")

            coords_train, vals_train, coords_test, vals_test = sample_regional_blocked(
                elevation, elev_lons, elev_lats, region_bounds, n_samples=n_samples, seed=seed
            )

            for act in ACTIVATIONS_EXP4:
                kwargs = {'n_knots': 15, 'init': 'relu'} if act == 'spline' else None

                enc = RawCoordinateEncoder(
                    activation_type=act,
                    activation_kwargs=kwargs
                )

                res = train_model(
                    f'raw_{region_name}_n{n_samples}_seed{seed}_{act}',
                    enc,
                    coords_train, vals_train,
                    coords_test, vals_test,
                    task_type='elevation',
                    epochs=EPOCHS,
                    verbose=False
                )

                res['seed'] = seed
                res['activation'] = act
                res['n_samples'] = n_samples
                res['region'] = region_name
                res['terrain'] = region_bounds['terrain']
                res['encoding'] = 'raw'

                results_exp4.append(res)
                print(f"{act}={res['r2']:.4f} ", end="")

            print()

df_exp4 = pd.DataFrame(results_exp4)

print("\n" + "="*80)
print("EXPERIMENT 4: TERRAIN EFFECT ANALYSIS")
print("="*80)

for region_name in REGIONS.keys():
    region_data = df_exp4[df_exp4['region'] == region_name]
    terrain = region_data['terrain'].iloc[0]

    print(f"\n--- {region_name.upper()} ({terrain}) ---")

    for n_samples in REGIONAL_SAMPLE_SIZES:
        subset = region_data[region_data['n_samples'] == n_samples]
        relu_r2s = subset[subset['activation'] == 'relu']['r2'].values
        spline_r2s = subset[subset['activation'] == 'spline']['r2'].values

        print(f"\nN = {n_samples:,}:")
        compare_activations(relu_r2s, spline_r2s, f"RAW COORDS - {region_name} (N={n_samples})")

print("\n⚠️  Compare to NB21 Exp 3 (with SH) to see if terrain effects differ")
print("="*80)

EXPERIMENT 4: REGIONAL MULTI-SEED (Raw Coordinates)

Region: ASIA_HIMALAYAS (mountainous)

  Sample Size: 10,000
    Seed 42: relu=0.4073 spline=0.4913 
    Seed 43: relu=0.9195 spline=0.9668 
    Seed 44: relu=0.8661 spline=0.8902 
    Seed 45: relu=0.0874 spline=0.6887 
    Seed 46: relu=0.8044 spline=0.9094 
    Seed 47: relu=0.5527 spline=0.7319 
    Seed 48: relu=0.8940 spline=0.9061 
    Seed 49: relu=0.7195 spline=0.8201 
    Seed 50: relu=0.6340 spline=0.8449 
    Seed 51: relu=0.8038 spline=0.8616 

  Sample Size: 20,000
    Seed 42: relu=0.5273 spline=0.6788 
    Seed 43: relu=0.9324 spline=0.9607 
    Seed 44: relu=0.8722 spline=0.8121 
    Seed 45: relu=0.4261 spline=0.6123 
    Seed 46: relu=0.8133 spline=0.8857 
    Seed 47: relu=0.6128 spline=0.7071 
    Seed 48: relu=0.9283 spline=0.9166 
    Seed 49: relu=0.7759 spline=0.8657 
    Seed 50: relu=0.7017 spline=0.8614 
    Seed 51: relu=0.8478 spline=0.9024 

Region: AFRICA_SAHARA (flat)

  Sample Size: 10,000
    Seed 42

---
## Experiment 5: Extended Training (Raw Coords)

**Question**: Convergence validation for raw coordinates

In [15]:
print("="*80)
print("EXPERIMENT 5: CONVERGENCE ANALYSIS (200 Epochs, Raw Coords)")
print("="*80)

SEEDS_EXP5 = range(42, 47)  # 5 seeds
ACTIVATIONS_EXP5 = ['relu', 'spline']
EPOCHS_CONV = 200

results_exp5 = []

for seed in SEEDS_EXP5:
    print(f"\nSeed {seed}:")

    coords_train, vals_train, coords_test, vals_test = sample_global_blocked(
        elevation, elev_lons, elev_lats, n_samples=15000, seed=seed
    )

    for act in ACTIVATIONS_EXP5:
        print(f"  {act.upper()}...", end=" ")

        kwargs = {'n_knots': 15, 'init': 'relu'} if act == 'spline' else None

        enc = RawCoordinateEncoder(
            activation_type=act,
            activation_kwargs=kwargs
        )

        res = train_model(
            f'raw_conv_seed{seed}_{act}',
            enc,
            coords_train, vals_train,
            coords_test, vals_test,
            task_type='elevation',
            epochs=EPOCHS_CONV,
            verbose=False,
            track_epochs=True
        )

        res['seed'] = seed
        res['activation'] = act
        res['encoding'] = 'raw'

        results_exp5.append(res)
        print(f"Best R²={res['r2']:.4f}")

print("\n" + "="*80)
print("CONVERGENCE SUMMARY")
print("="*80)

for act in ACTIVATIONS_EXP5:
    act_results = [r for r in results_exp5 if r['activation'] == act]

    r2_at_100 = []
    r2_at_200 = []

    for res in act_results:
        history = res['r2_history']
        r2_at_100.append(max(history[:100]))
        r2_at_200.append(max(history))

    improvement = np.array(r2_at_200) - np.array(r2_at_100)
    imp_stats = compute_stats(improvement)

    print(f"\n{act.upper()}:")
    print(f"  R² at epoch 100: {np.mean(r2_at_100):.4f} ± {np.std(r2_at_100):.4f}")
    print(f"  R² at epoch 200: {np.mean(r2_at_200):.4f} ± {np.std(r2_at_200):.4f}")
    print(f"  Improvement:     {imp_stats['mean']:.4f} ± {imp_stats['std']:.4f}")

    if imp_stats['mean'] > 0.01:
        print("  ⚠️  UNDERTRAINING: Models still improving")
    else:
        print("  ✅ CONVERGED: 100 epochs sufficient")

print("="*80)

EXPERIMENT 5: CONVERGENCE ANALYSIS (200 Epochs, Raw Coords)

Seed 42:
  RELU... Best R²=0.8666
  SPLINE... Best R²=0.8941

Seed 43:
  RELU... Best R²=0.8583
  SPLINE... Best R²=0.8892

Seed 44:
  RELU... Best R²=0.8608
  SPLINE... Best R²=0.8967

Seed 45:
  RELU... Best R²=0.8655
  SPLINE... Best R²=0.9001

Seed 46:
  RELU... Best R²=0.8659
  SPLINE... Best R²=0.9076

CONVERGENCE SUMMARY

RELU:
  R² at epoch 100: 0.8343 ± 0.0079
  R² at epoch 200: 0.8634 ± 0.0033
  Improvement:     0.0291 ± 0.0071
  ⚠️  UNDERTRAINING: Models still improving

SPLINE:
  R² at epoch 100: 0.8747 ± 0.0044
  R² at epoch 200: 0.8975 ± 0.0062
  Improvement:     0.0228 ± 0.0044
  ⚠️  UNDERTRAINING: Models still improving


---
## Final Synthesis: Raw vs SH Comparison

**Critical Analysis: Does SH mask learned activation benefits?**

In [16]:
print("="*80)
print("FINAL SYNTHESIS: RAW vs SH ENCODING")
print("="*80)

print("\n" + "="*80)
print("EXPERIMENT 1 & 2: GLOBAL TASKS SUMMARY")
print("="*80)

# Elevation
raw_relu_elev = df_exp1_elev[df_exp1_elev['activation'] == 'relu']['r2'].values
raw_spline_elev = df_exp1_elev[df_exp1_elev['activation'] == 'spline']['r2'].values
raw_adv_elev = 100 * (raw_spline_elev - raw_relu_elev) / raw_relu_elev
raw_adv_elev_stats = compute_stats(raw_adv_elev)

# Population
raw_relu_pop = df_exp2_pop[df_exp2_pop['activation'] == 'relu']['r2'].values
raw_spline_pop = df_exp2_pop[df_exp2_pop['activation'] == 'spline']['r2'].values
raw_adv_pop = 100 * (raw_spline_pop - raw_relu_pop) / raw_relu_pop
raw_adv_pop_stats = compute_stats(raw_adv_pop)

print("\n--- ELEVATION TASK ---")
print(f"\nRaw+ReLU:   {raw_relu_elev.mean():.4f} ± {raw_relu_elev.std():.4f}")
print(f"Raw+Spline: {raw_spline_elev.mean():.4f} ± {raw_spline_elev.std():.4f}")
print(f"Spline Advantage: {raw_adv_elev_stats['mean']:+.2f}% ± {raw_adv_elev_stats['std']:.2f}%")
print(f"95% CI: [{raw_adv_elev_stats['ci_low']:+.2f}%, {raw_adv_elev_stats['ci_high']:+.2f}%]")

print("\n⚠️  Compare to NB21 SH+acts results:")
print("   SH+ReLU:   [FROM NB21]")
print("   SH+Spline: [FROM NB21]")
print("   Advantage: [FROM NB21] (NB19 showed +0.36%)")

print("\n--- POPULATION TASK ---")
print(f"\nRaw+ReLU:   {raw_relu_pop.mean():.4f} ± {raw_relu_pop.std():.4f}")
print(f"Raw+Spline: {raw_spline_pop.mean():.4f} ± {raw_spline_pop.std():.4f}")
print(f"Spline Advantage: {raw_adv_pop_stats['mean']:+.2f}% ± {raw_adv_pop_stats['std']:.2f}%")
print(f"95% CI: [{raw_adv_pop_stats['ci_low']:+.2f}%, {raw_adv_pop_stats['ci_high']:+.2f}%]")

print("\n⚠️  Compare to NB21b SH+acts results:")
print("   SH+ReLU:   [FROM NB21b]")
print("   SH+Spline: [FROM NB21b]")
print("   Advantage: [FROM NB21b] (NB19b showed -0.64%)")

print("\n" + "="*80)
print("KEY INTERPRETATIONS")
print("="*80)

print("\n1. Does SH add value (vs raw)?")
print("   Compare: SH+ReLU vs Raw+ReLU")
print("   [FILL IN AFTER NB21/21b COMPLETE]")

print("\n2. Does SH mask spline gains?")
print("   Compare: Raw spline advantage vs SH spline advantage")
print("   [FILL IN AFTER NB21/21b COMPLETE]")

print("\n3. Can splines win without SH?")
if raw_adv_elev_stats['ci_low'] > 1.0:
    print("   ✅ YES (Elevation): Spline advantage > 1%, 95% CI excludes zero")
elif raw_adv_elev_stats['ci_high'] < -1.0:
    print("   ❌ NO (Elevation): ReLU wins significantly")
else:
    print("   ⚠️  INCONCLUSIVE (Elevation): Effect too small or inconsistent")

if raw_adv_pop_stats['ci_low'] > 1.0:
    print("   ✅ YES (Population): Spline advantage > 1%, 95% CI excludes zero")
elif raw_adv_pop_stats['ci_high'] < -1.0:
    print("   ❌ NO (Population): ReLU wins significantly")
else:
    print("   ⚠️  INCONCLUSIVE (Population): Effect too small or inconsistent")

print("\n" + "="*80)
print("NEXT STEPS")
print("="*80)

print("\nOnce NB21/21b complete:")
print("1. Fill in [FROM NB21/21b] placeholders above")
print("2. Create comparative visualization: Raw vs SH for each activation")
print("3. Calculate SH encoding 'overhead': (SH params - Raw params) vs R² gain")
print("4. Decide:")
print("   - If Raw+Spline wins: Investigate why SH masks benefit")
print("   - If SH+acts >> Raw+acts: SH encoding is essential")
print("   - If both show ReLU wins: Write up 'simplicity bias sufficient'")

print("\n" + "="*80)

FINAL SYNTHESIS: RAW vs SH ENCODING

EXPERIMENT 1 & 2: GLOBAL TASKS SUMMARY

--- ELEVATION TASK ---

Raw+ReLU:   0.8222 ± 0.0121
Raw+Spline: 0.8721 ± 0.0099
Spline Advantage: +6.09% ± 2.10%
95% CI: [+4.58%, +7.59%]

⚠️  Compare to NB21 SH+acts results:
   SH+ReLU:   [FROM NB21]
   SH+Spline: [FROM NB21]
   Advantage: [FROM NB21] (NB19 showed +0.36%)

--- POPULATION TASK ---

Raw+ReLU:   0.5375 ± 0.0409
Raw+Spline: 0.5832 ± 0.0332
Spline Advantage: +8.68% ± 2.92%
95% CI: [+6.59%, +10.77%]

⚠️  Compare to NB21b SH+acts results:
   SH+ReLU:   [FROM NB21b]
   SH+Spline: [FROM NB21b]
   Advantage: [FROM NB21b] (NB19b showed -0.64%)

KEY INTERPRETATIONS

1. Does SH add value (vs raw)?
   Compare: SH+ReLU vs Raw+ReLU
   [FILL IN AFTER NB21/21b COMPLETE]

2. Does SH mask spline gains?
   Compare: Raw spline advantage vs SH spline advantage
   [FILL IN AFTER NB21/21b COMPLETE]

3. Can splines win without SH?
   ✅ YES (Elevation): Spline advantage > 1%, 95% CI excludes zero
   ✅ YES (Population)

---
## Optional: Save Results

In [17]:
# Optional: Save to Google Drive
if 'COLAB_GPU' in os.environ:
    save_dir = '/content/drive/MyDrive/learned_activation_results/nb21c/'
    os.makedirs(save_dir, exist_ok=True)

    df_exp1_elev.to_csv(f'{save_dir}exp1_raw_elevation_multiseed.csv', index=False)
    df_exp2_pop.to_csv(f'{save_dir}exp2_raw_population_multiseed.csv', index=False)
    df_exp3.to_csv(f'{save_dir}exp3_raw_sample_size.csv', index=False)
    df_exp4.to_csv(f'{save_dir}exp4_raw_regional_multiseed.csv', index=False)

    print(f"✅ Results saved to: {save_dir}")
else:
    print("Not on Colab, skip Drive save")

✅ Results saved to: /content/drive/MyDrive/learned_activation_results/nb21c/
